# Backend-agnostic fixed-point benchmark

This notebook separates **planning** from **execution**. Optyx first compiles a finite feedback process with `at_time(n_steps).double().to_tensor()`. The output is a `discopy.tensor.Diagram`; exact tensor functors, exact Quimb and compressed Quimb all consume that same intermediate representation. We estimate an exact contraction path before running any contraction, then compare fixed-point methods and backends.

In [ ]:
from importlib.util import find_spec
from time import perf_counter

import numpy as np

from optyx import qubits
from optyx.channel import frobenius_distance, qubit
from optyx.core.backends import DiscopyBackend, QuimbBackend

process = (qubits.X(1, 1, 0.25) >> qubits.Z(1, 2)).feedback(
    mem=qubit, initial_state=qubits.Ket(0))

## 1. Estimate before contracting

`contraction_info` searches for an exact path and reports its arithmetic cost and largest intermediate; it does not execute the contraction. The seconds column is only a hardware-independent rule of thumb based on an assumed sustained throughput. Python overhead, memory traffic, compilation and accelerator transfer can dominate small networks.

In [ ]:
def compile_network(diagram, n_steps):
    return diagram.at_time(n_steps).double().to_tensor()


def estimate_exact_contraction(tensor_diagram, assumed_gflops=10.0):
    network = tensor_diagram.to_quimb()
    info = network.contraction_info(
        optimize="greedy",
        output_inds=sorted(network.outer_inds()))
    flops = float(info.opt_cost)
    elements = int(info.largest_intermediate)
    return {
        "tensors": len(network.tensor_map),
        "flops": flops,
        "largest_complex128_bytes": 16 * elements,
        "heuristic_seconds": flops / (assumed_gflops * 1e9),
    }


compiled = {n: compile_network(process, n) for n in (2, 4, 8)}
plans = {n: estimate_exact_contraction(network)
         for n, network in compiled.items()}
plans

Inspect `plans` before continuing. If the largest exact intermediate does not fit in memory, do not execute the exact cases at that depth. A compressed contraction can help there, but its real cost also depends on `chi`, tensor geometry and the truncation sequence, so the exact path estimate is a warning signal rather than a compressed-runtime prediction.

## 2. Benchmark methods and backends

The eigen method is a separate small-memory reference. Every power case compiles the same depth-four network and changes only `TensorBackend`. JAX and PyTorch are optional; unavailable or unsupported local configurations are reported instead of aborting the comparison. Timings are cold-start wall times and therefore include array-library setup.

In [ ]:
def normalised_density(result):
    matrix = np.asarray(result.density_matrix)
    return matrix / np.trace(matrix)


def time_case(label, run, reference=None):
    started = perf_counter()
    try:
        result = run()
        row = {"case": label, "seconds": perf_counter() - started}
        if reference is not None:
            row["distance_to_eigen"] = frobenius_distance(
                normalised_density(result), reference)
        return row
    except Exception as error:  # Optional runtime or device unsupported.
        return {"case": label, "error": repr(error)}


eigen_result = process.fix(method="eigen")
reference = normalised_density(eigen_result)
cases = [
    ("eigen", lambda: process.fix(method="eigen")),
    ("power / tensor.Functor / NumPy",
     lambda: process.fix(n_steps=4, chi=4,
                         backend=DiscopyBackend("numpy"))),
    ("power / Quimb exact",
     lambda: process.fix(n_steps=4, chi=4, backend=QuimbBackend())),
    ("power / Quimb compressed / chi=4",
     lambda: process.fix(n_steps=4, chi=4,
                         backend=QuimbBackend.compressed())),
]
if find_spec("jax"):
    cases.append(("power / tensor.Functor / JAX",
                  lambda: process.fix(
                      n_steps=4, chi=4, backend=DiscopyBackend("jax"))))
if find_spec("torch"):
    cases.append(("power / tensor.Functor / PyTorch",
                  lambda: process.fix(
                      n_steps=4, chi=4,
                      backend=DiscopyBackend("pytorch"))))

benchmark = [time_case(label, run, reference) for label, run in cases]
benchmark

## 3. Where approximation helps

There are three independent accuracy controls:

1. **Finite-time error**: increase `n_steps` for every power backend and compare normalised density matrices. This is physical convergence from `initial_state`, not tensor truncation.
2. **Bond-compression error**: only compressed Quimb truncates intermediate bonds. Repeat at increasing `chi` and compare with an exact result at a feasible depth. Exact NumPy, JAX, PyTorch and Quimb ignore `chi`.
3. **Optical cutoff error**: the eigen method truncates every optical memory mode to `cutoff`. Increase it separately; this is unrelated to `chi`.

Compression is useful when the exact `largest_complex128_bytes` estimate approaches available memory. It is unnecessary on small networks, where optimiser and device-startup overhead can be larger than the contraction itself. Report both runtime and distance to the eigen or feasible exact reference; a fast result without a convergence sweep is not an accuracy claim.